In [0]:
from pyspark.sql import functions as F

s = '''{
  "order_id": 5001,
  "placed_at": "2026-09-30T14:25:00",
  "customer": {"id": "C42", "name": "Jonas Weber",
               "address": {"city": "Munich", "zip": "80331"}},
  "items": [
    {"sku": "A100", "qty": 2, "price": 19.99, "tags": ["new", "eco"]},
    {"sku": "B200", "qty": 1, "price": 5.5,  "tags": []}
  ],
  "payment": {"method": "card", "paid": true}
}'''

raw = spark.createDataFrame([(s,)], ["v"])   # one row, one string column

In [0]:
display(raw)

In [0]:
raw_df = spark.read.json(raw)
raw_df.printSchema()

In [0]:
df = (raw_df
  .withColumn("order_id", F.col("order_id").cast("int"))
  .withColumn("placed_at", F.to_timestamp("placed_at"))
  .withColumn("it", F.explode("items"))          # one explode, one struct per item
  .select(
      "order_id", "placed_at",
      F.col("customer.id").alias("customer_id"),
      F.col("customer.name").alias("customer_name"),
      F.col("customer.address.city").alias("city"),
      F.col("customer.address.zip").alias("zip"),
      F.col("it.sku").alias("sku"),
      F.col("it.qty").alias("qty"),
      F.col("it.price").alias("price"),
      (F.col("it.qty") * F.col("it.price")).alias("line_total"),
       F.array_join("it.tags", ",").alias("tags"),
    #   F.col("it.tag").alias("tags"),
      F.col("payment.method").alias("payment_method"),
      F.col("payment.paid").alias("paid"),
  ))

print(df.count())   # must be 2
df.show()
df.printSchema()